# Assignment 1: Weather Agent Debugging

## 1. Check Config and Helper Functions

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("weather_agent"))

from components.config import config
from components.helper_functions import (
    classify_temperature,
    get_weather_description,
    get_greeting,
    format_local_time,
)

print("Location URL:", config.LOCATION_API_URL)
print("Weather URL:", config.WEATHER_API_BASE_URL)

for temp in [-2.0, 8.0, 14.0, 22.0, 29.0, 35.0]:
    print(f"{temp} C -> {classify_temperature(temp)}")

print("Day greeting:", get_greeting(1))
print("Night greeting:", get_greeting(0))
print("Code 2:", get_weather_description(2))
print("Local time (+0530):", format_local_time("2026-10-23T04:00", "+0530"))

Location URL: https://ipapi.co/json/
Weather URL: https://api.open-meteo.com/v1/forecast
-2.0 C -> cold
8.0 C -> cold
14.0 C -> cool
22.0 C -> comfortable
29.0 C -> warm
35.0 C -> hot
Day greeting: Good morning
Night greeting: Good evening
Code 2: Partly cloudy
Local time (+0530): 04:00 UTC | 09:30 (UTC+0530)

## 2. Test the Agent Across Different Scenarios

In [2]:
from unittest.mock import patch, MagicMock
from graph import weather_agent
from components.state import WeatherAgentState

test_cases = [
    {
        "name": "Aarav",
        "location": {
            "city": "Bengaluru",
            "region": "Karnataka",
            "country_name": "India",
            "latitude": 12.9716,
            "longitude": 77.5946,
            "utc_offset": "+0530",
            "timezone": "Asia/Kolkata",
        },
        "weather": {
            "current_weather_units": {"temperature": "°C", "windspeed": "km/h"},
            "current_weather": {
                "time": "2026-10-23T04:00",
                "temperature": 23.4,
                "windspeed": 11.2,
                "winddirection": 95,
                "is_day": 1,
                "weathercode": 2,
            },
        },
    },
    {
        "name": "Katrin",
        "location": {
            "city": "Reykjavik",
            "region": "Capital Region",
            "country_name": "Iceland",
            "latitude": 64.1466,
            "longitude": -21.9426,
            "utc_offset": "+00:00",
            "timezone": "Atlantic/Reykjavik",
        },
        "weather": {
            "current_weather_units": {"temperature": "°C", "windspeed": "km/h"},
            "current_weather": {
                "time": "2026-10-23T22:15",
                "temperature": -2.8,
                "windspeed": 28.5,
                "winddirection": 30,
                "is_day": 0,
                "weathercode": 73,
            },
        },
    },
    {
        "name": "Zayd",
        "location": {
            "city": "Dubai",
            "region": "Dubai",
            "country_name": "United Arab Emirates",
            "latitude": 25.2048,
            "longitude": 55.2708,
            "utc_offset": "+04:00",
            "timezone": "Asia/Dubai",
        },
        "weather": {
            "current_weather_units": {"temperature": "°C", "windspeed": "km/h"},
            "current_weather": {
                "time": "2026-10-23T09:30",
                "temperature": 36.2,
                "windspeed": 18.0,
                "winddirection": 310,
                "is_day": 1,
                "weathercode": 0,
            },
        },
    },
]

for case in test_cases:
    def fake_get(url, params=None, timeout=None):
        resp = MagicMock()
        resp.raise_for_status = MagicMock()
        if "ipapi.co" in url:
            resp.json.return_value = case["location"]
        else:
            resp.json.return_value = case["weather"]
        return resp

    with patch("components.nodes.requests.get", side_effect=fake_get):
        state = WeatherAgentState(
            name=case["name"],
            location_data=None,
            weather_data=None,
            weather_info=None,
        )
        result = weather_agent.invoke(state)
        print(result["weather_info"])
        print()

Time: 04:00 UTC | 09:30 (UTC+0530)

Good morning, Aarav!

Your current location: Bengaluru, Karnataka, India

Current weather conditions:
- Partly cloudy
- Temperature: 23.4°C (comfortable)
- Wind: 11.2 km/h

Time: 22:15 UTC | 22:15 (UTC+00:00)

Good evening, Katrin!

Your current location: Reykjavik, Capital Region, Iceland

Current weather conditions:
- Moderate snow fall
- Temperature: -2.8°C (cold)
- Wind: 28.5 km/h

Time: 09:30 UTC | 13:30 (UTC+04:00)

Good morning, Zayd!

Your current location: Dubai, Dubai, United Arab Emirates

Current weather conditions:
- Clear sky
- Temperature: 36.2°C (hot)
- Wind: 18.0 km/h


## 3. Error Handling Check

In [3]:
import main
import requests

def fake_bad_location(url, params=None, timeout=None):
    resp = MagicMock()
    resp.raise_for_status = MagicMock()
    resp.json.return_value = {"city": "TestCity"}
    return resp

with patch("builtins.input", return_value="Alex"), patch("components.nodes.requests.get", side_effect=fake_bad_location):
    main.main()

def fake_timeout(url, params=None, timeout=None):
    raise requests.exceptions.ConnectTimeout("Request timed out")

with patch("builtins.input", return_value="Alex"), patch("components.nodes.requests.get", side_effect=fake_timeout):
    main.main()


Error: Invalid location data received: Missing required field: region
Please check your internet connection and try again.

Error: Failed to fetch location data: Request timed out
Please check your internet connection and try again.